In [ ]:
from pathlib import Path
import pandas as pd

raw = Path(...) #vous remplacez les ... par le chemin vers le dossier raw
markets = ['DK1', 'DE', 'DK2', 'NO2', 'SE4']
summary = []

for market in markets:
    files = sorted((raw / market / 'Prices').glob('*.csv'))
    print(f'\n{market}: {len(files)} files')

    for file in files:
        data = pd.read_csv(file, nrows=2, sep=None, engine='python', encoding='utf-8-sig')
        print(file.name, '|', list(data.columns))
        summary.append({'market': market, 'file': file.name,
                        'columns': ', '.join(data.columns)})

pd.DataFrame(summary).to_csv(raw.parent / 'raw_price_discovery.csv', index=False)
print('\nDiscovery saved to raw_price_discovery.csv')



DK1: 11 files
DK1_2015-2016_Prices.csv | ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Price (EUR/MWh)']
DK1_2016-2017_Prices.csv | ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Price (EUR/MWh)']
DK1_2017-2018_Prices.csv | ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Price (EUR/MWh)']
DK1_2018-2019_Prices.csv | ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Price (EUR/MWh)']
DK1_2019-2020_Prices.csv | ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Price (EUR/MWh)']
DK1_2020-2021_Prices.csv | ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Price (EUR/MWh)']
DK1_2021-2022_Prices.csv | ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Pr

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

root = Path(...) #vous remplacez les ... par le chemin vers le dossier root
markets = ['DK1', 'DE', 'DK2', 'NO2', 'SE4']
hedges = markets[1:]
price_column = 'Day-ahead Price (EUR/MWh)'
output = root / 'hedge_results'
output.mkdir(exist_ok=True)


def load_prices(market):
    folder = root / 'ElectricityPrices' / market / 'Prices'
    tables = []

    for file in sorted(folder.glob('*.csv')):
        data = pd.read_csv(file, sep=None, engine='python', encoding='utf-8-sig', dtype=str)
        time_column = 'MTU (UTC)' if market in ['DK1', 'DK2'] else 'MTU (CET/CEST)'

        start = data[time_column].str.split(' - ', regex=False).str[0]
        time = pd.to_datetime(start, format='%d/%m/%Y %H:%M:%S', errors='coerce')
        if market in ['DK1', 'DK2']:
            time = time.dt.tz_localize('UTC')
        else:
            time = time.dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='NaT').dt.tz_convert('UTC')

        values = data[price_column].str.strip().str.replace(' ', '', regex=False)
        has_comma = values.str.contains(',', regex=False, na=False)
        values.loc[has_comma] = values.loc[has_comma].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        prices = pd.to_numeric(values, errors='coerce')
        tables.append(pd.DataFrame({'time': time, market: prices}).dropna())

    if not tables:
        raise FileNotFoundError(f'No CSV files found for {market}: {folder}')
    results = []
    result = pd.concat(tables).sort_values('time')
    conflicts = result.groupby('time')[market].nunique()
    if (conflicts > 1).any():
        raise ValueError(f'Conflicting prices at repeated timestamps in {market}')
    
    result = result.drop_duplicates('time').set_index('time')
    print(f'{market}: {len(result):,} observations')
    return result


prices = pd.concat([load_prices(market) for market in markets], axis=1, join='inner').dropna().sort_index()
prices.to_csv(output / 'aligned_day_ahead_prices.csv')

hourly = prices.index.to_series().diff() == pd.Timedelta(hours=1)
changes = prices.diff().loc[hourly].dropna()
if len(changes) < 300:
    raise ValueError('Too few aligned hourly observations')

cut = int(len(changes) * 0.7)
train = changes.iloc[:cut]
test = changes.iloc[cut:]

X = train[hedges].to_numpy()
y = train['DK1'].to_numpy()


def variance(weights):
    return np.var(y - X @ weights, ddof=1)


fit = minimize(variance, x0=np.full(4, 0.2), method='SLSQP',
               bounds=[(0, 1)] * 4,
               constraints={'type': 'ineq', 'fun': lambda w: 1 - sum(w)},
               options={'maxiter': 3000, 'ftol': 1e-11})
if not fit.success:
    raise RuntimeError(fit.message)

weights = fit.x
weight_table = pd.DataFrame({'market': hedges, 'weight': weights})
weight_table.to_csv(output / 'optimal_weights.csv', index=False)

results = []
for name, data in [('train', train), ('test', test)]:
    unhedged = data['DK1']
    hedged = unhedged - data[hedges].to_numpy() @ weights
    v0, v1 = unhedged.var(), hedged.var()

    results.append({'sample': name, 'n': len(data),
                    'unhedged_variance': v0, 'hedged_variance': v1,
                    'unhedged_std': np.sqrt(v0), 'hedged_std': np.sqrt(v1),
                    'variance_reduction_pct': 100 * (1 - v1 / v0)})

    if name == 'test':
        comparison = pd.DataFrame({'unhedged_change': unhedged,
                                   'hedged_proxy_change': hedged}, index=data.index)
        comparison.to_csv(output / 'test_changes.csv')
        ax = comparison.rolling(24, min_periods=24).sum().plot(figsize=(11, 4))
        ax.set(title='Rolling 24-observation sum of hourly price changes', ylabel='EUR/MWh')
        plt.tight_layout()
        plt.savefig(output / 'rolling_24_observations.png', dpi=160)
        plt.close()

risk_table = pd.DataFrame(results)
risk_table.to_csv(output / 'risk_comparison.csv', index=False)
print('\nOptimal weights:\n', weight_table.to_string(index=False))
print('\nRisk comparison:\n', risk_table.to_string(index=False))
print('\nResults saved in:', output)
print('Spot-price proxy hedge only, not a futures trading backtest.')


DK1: 96,144 observations
DE: 63,561 observations
DK2: 96,144 observations
NO2: 87,610 observations
SE4: 87,610 observations

Optimal weights:
 market   weight
    DE 0.450390
   DK2 0.407056
   NO2 0.137922
   SE4 0.004633

Risk comparison:
 sample     n  unhedged_variance  hedged_variance  unhedged_std  hedged_std  variance_reduction_pct
 train 44272         315.772138        74.540005     17.769979    8.633655               76.394369
  test 18974         373.000189        54.289351     19.313213    7.368131               85.445222

Results saved in: C:\Users\Asus-U\Desktop\M2 FTD\Master Thesis - Copie\case b\hedge_results
Spot-price proxy hedge only, not a futures trading backtest.
